# TP3 du cours de vision par ordinateur

In [1]:
import torch
import torchvision
import sklearn
import numpy as np
import matplotlib.pyplot as plt
import kagglehub

c:\Users\loicr\Cours\UQAC\Session_Hiver\Vision\Env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Importer un modèle pré-entraîné (EfficientNet-B0)
model = torchvision.models.efficientnet_b0(pretrained=True, progress = True)

c:\Users\loicr\Cours\UQAC\Session_Hiver\Vision\Env\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\loicr\Cours\UQAC\Session_Hiver\Vision\Env\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
# Download latest version
path = kagglehub.dataset_download("phucthaiv02/butterfly-image-classification")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\loicr\.cache\kagglehub\datasets\phucthaiv02\butterfly-image-classification\versions\3


In [ ]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Transformations pour EfficientNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ButterflyDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        # Encoder les labels en entiers
        self.classes = sorted(self.df["label"].unique())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["filename"])
        image = Image.open(img_path).convert("RGB")
        label = self.class_to_idx[row["label"]]
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = ButterflyDataset(os.path.join(path, "Training_set.csv"), os.path.join(path, "train"), transform)

train_dataset, test_dataset = sklearn.model_selection.train_test_split(train_dataset, test_size=0.2, random_state=42)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

num_classes = len(train_dataset.classes)
print(f"Classes: {num_classes}")
print(f"Train: {len(train_dataset)} images | Test: {len(test_dataset)} images")

In [ ]:
# Adapter le classifier au nombre de classes du dataset
model.classifier[1] = torch.nn.Linear(in_features=1280, out_features=num_classes)
print(f"Classifier adapté pour {num_classes} classes")
print(model)

In [ ]:
# Finetune du modele

from torch import nn, optim
from torch.amp import autocast, GradScaler

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
criterion = torch.nn.CrossEntropyLoss()

# Geler les couches convolutives
for module in model.features:
    for param in module.parameters():
        param.requires_grad = False

num_epochs = 10
log_interval = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

scaler = GradScaler()

for epoch in range(num_epochs):
    model.train()
    for batch_idx, (data, targets) in enumerate(train_loader):
        data, targets = data.to(device), targets.to(device)

        optimizer.zero_grad()

        with autocast(device_type=device.type):
            outputs = model(data)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if batch_idx % log_interval == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

C:\Users\loicr\AppData\Local\Temp\ipykernel_19084\3659332935.py:18: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
c:\Users\loicr\Cours\UQAC\Session_Hiver\Vision\Env\Lib\site-packages\torch\cuda\amp\grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


AttributeError: module 'torch.nn' has no attribute 'train_loader'